In [1]:
import jax
import jax.numpy as jnp
import matplotlib.colors
import matplotlib.patches
import numpy as np
from matplotlib import pyplot as plt

from xxm.core.discrete.emissions_ar import ARGaussianEmissions, flatten_ar_coefficients
from xxm.core.discrete.utils import match_states_by_conditional_mean
from xxm.hmm.core import DiscreteInitialModel, DiscreteTransitionModel, Model
from xxm.hmm.inference import inference_exact
from xxm.hmm.init import initialize_arhmm_gaussian
from xxm.hmm.learning import fit_em
from xxm.stats.gaussian import Affine, LinearGaussian

In [2]:
STATES_CMAP = matplotlib.colors.ListedColormap(['tab:blue', 'tab:orange', 'tab:green'])


def make_true_model() -> Model:
    return Model(
        initial=DiscreteInitialModel(
            initial_probs=jnp.array([0.4, 0.3, 0.3]),
        ),
        transitions=DiscreteTransitionModel(
            transition_probs=jnp.array(
                [
                    [0.97, 0.02, 0.01],
                    [0.02, 0.97, 0.01],
                    [0.02, 0.02, 0.96],
                ]
            )
        ),
        emissions=ARGaussianEmissions(
            LinearGaussian(
                Affine(
                    coefficients=flatten_ar_coefficients(
                        jnp.array(
                            [
                                # clockwise rotation
                                [
                                    [
                                        [0.92, 0.32],
                                        [-0.32, 0.92],
                                    ]
                                ],
                                # counter-clockwise rotation
                                [
                                    [
                                        [0.92, -0.32],
                                        [0.32, 0.92],
                                    ]
                                ],
                                # anisotropic dynamics
                                [
                                    [
                                        [0.97, 0.00],
                                        [0.00, 0.65],
                                    ]
                                ],
                            ]
                        )
                    ),  # (K=3, L=1, N=2, N=2)
                    bias=jnp.zeros((3, 2)),
                ),
                covariance=jnp.array(
                    [
                        [[0.05, 0.00], [0.00, 0.05]],
                        [[0.05, 0.00], [0.00, 0.05]],
                        [[0.05, 0.00], [0.00, 0.05]],
                    ]
                ),
            ),
        ),
    )


true_model = make_true_model()

true_states, observations = true_model.sample(
    num_steps=1000,
    key=jax.random.key(0),
)

In [ ]:
def plot_sequence_2d(
    observations: jnp.ndarray,
    states: jnp.ndarray,
) -> None:
    observations = jnp.asarray(observations)
    states = jnp.asarray(states)

    f, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)

    ax.plot(
        observations[:, 0],
        observations[:, 1],
        linewidth=0.5,
        alpha=0.3,
    )

    ax.scatter(
        observations[:, 0],
        observations[:, 1],
        c=states,
        s=12,
        cmap=STATES_CMAP,
    )

    ax.set_xlabel('Observation 1')
    ax.set_ylabel('Observation 2')
    ax.set_aspect('equal')


plot_sequence_2d(observations, true_states)

In [ ]:
def plot_linear_dynamics(
    ax,
    matrix,
    bias=None,
    xlim=(-3, 3),
    ylim=(-3, 3),
    num_points=15,
    **kwargs,
):
    """Plot the expected displacement field of 2D linear dynamics."""
    matrix = np.asarray(matrix)

    if bias is None:
        bias = np.zeros(2)
    else:
        bias = np.asarray(bias)

    x = np.linspace(*xlim, num_points)  # type: ignore
    y = np.linspace(*ylim, num_points)  # type: ignore
    xx, yy = np.meshgrid(x, y)

    points = np.stack([xx, yy], axis=-1)  # (..., 2)

    next_points = points @ matrix.T + bias
    displacement = next_points - points

    ax.quiver(
        xx,
        yy,
        displacement[..., 0],
        displacement[..., 1],
        **kwargs,
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.set_xlabel(r'$y_1$')
    ax.set_ylabel(r'$y_2$')


def plot_model_dynamics(model: Model) -> None:
    """Plot the expected displacement field of a 2D linear dynamics model."""

    fig, axs = plt.subplots(
        figsize=(8, 6),
        ncols=model.emissions.num_states,
        sharex='all',
        sharey='all',
        constrained_layout=True,
        squeeze=False,
    )

    axs = axs.ravel()

    for state, ax in enumerate(axs):
        plot_linear_dynamics(
            ax,
            matrix=model.emissions.model.affine.coefficients[state],
            bias=model.emissions.model.affine.bias[state],
        )

        ax.set_title(f'State {state}')


plot_model_dynamics(true_model)

In [ ]:
posterior = inference_exact(true_model, observations)

In [ ]:
def plot_sequence_1d(
    ax,
    observations: jnp.ndarray,
    states: jnp.ndarray,
) -> None:
    observations = jnp.asarray(observations)
    states = jnp.asarray(states)

    ax.imshow(
        states.reshape(-1, 1).T,
        aspect='auto',
        cmap=STATES_CMAP,
        vmin=0,
        vmax=2,
        alpha=0.25,
        extent=(0, len(states), 0, 1),
        transform=ax.get_xaxis_transform(),
    )

    ax.plot(observations.T[0], color='k')
    ax.plot(observations.T[1], color='xkcd:magenta')

    ax.set(xlabel='Time step', ylabel='Observations')


def plot_inferred_states(observations, true_states, posterior):
    f, axs = plt.subplots(nrows=2, constrained_layout=True, figsize=(8, 6))

    ax = axs[0]
    plot_sequence_1d(ax, observations, true_states)

    ax.set(
        title='True states',
    )

    ax = axs[1]
    plot_sequence_1d(ax, observations, posterior.state_marginals.argmax(axis=1))
    ax.set(
        title='Inferred posterior states',
    )


plot_inferred_states(observations, true_states, posterior)

In [ ]:
def plot_log_likelihood(log_likelihoods: jnp.ndarray) -> None:
    f, ax = plt.subplots(figsize=(6, 3))

    ax.plot(np.asarray(log_likelihoods))

    ax.set_xlabel('EM iteration')
    ax.set_ylabel('Log likelihood')


initial_model = initialize_arhmm_gaussian(
    observations=observations,
    num_states=true_model.num_states,
    key=jax.random.key(0),
    lag=1,
)


fit_em_jit = jax.jit(fit_em, static_argnames='num_iters')

learned_model, log_likelihoods = fit_em_jit(
    model=initial_model,
    observations=observations,
    num_iters=50,
)


plot_log_likelihood(log_likelihoods)

In [ ]:
permutation = match_states_by_conditional_mean(
    true_model.emissions.conditional(observations).mean,
    learned_model.emissions.conditional(observations).mean,
)

matched_model = learned_model.permute(permutation)

In [ ]:
def plot_model_dynamics_comparison(model0: Model, model1: Model) -> None:
    """Plot the expected displacement field of a 2D linear dynamics model."""

    assert model0.num_states == model1.num_states, 'Models must have the same number of states.'

    fig, axs = plt.subplots(
        figsize=(8, 6), ncols=model0.num_states, sharex='all', sharey='all', constrained_layout=True
    )

    for state, ax in enumerate(axs):
        plot_linear_dynamics(
            ax,
            matrix=model0.emissions.model.affine.coefficients[state],
            bias=model0.emissions.model.affine.bias[state],
            color='k',
        )

        plot_linear_dynamics(
            ax,
            matrix=model1.emissions.model.affine.coefficients[state],
            bias=model1.emissions.model.affine.bias[state],
            color='xkcd:magenta',
        )

        ax.set_title(f'State {state}')


plot_model_dynamics_comparison(true_model, matched_model)

In [ ]:
posterior = inference_exact(matched_model, observations)

In [ ]:
def plot_inferred_states(observations, true_states, posterior):
    f, axs = plt.subplots(nrows=2, constrained_layout=True, figsize=(8, 6))

    ax = axs[0]
    plot_sequence_1d(ax, observations, true_states)

    ax.set(
        title='True states',
    )

    ax = axs[1]
    plot_sequence_1d(ax, observations, posterior.state_marginals.argmax(axis=1))
    ax.set(
        title='Inferred posterior states',
    )


plot_inferred_states(observations, true_states, posterior)